<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/notebooks/pytorch_dl_testbed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Deep-Learning Testbed — Synthetic Entropy-Divergence Sandbox

**Status: experimental sandbox — not part of the production pipeline.** Local runs write to `notebooks/outputs/` (untracked); Kaggle GPU CI runs (see `notebooks/README.md`) write to `notebooks/results/`, which *is* committed.

This notebook is a **self-contained playground for testing deep-learning ideas in pure PyTorch**
against this project's core research question — *can internal uncertainty signals (token entropy,
divergence between stochastic passes) separate hallucinated from factual generations?* — without
touching any production module in this repo. Ideas that survive here get ported into the real
pipeline afterwards (see the integration checklist at the bottom).

## Research questions under test

| # | Question | Motivation |
|---|----------|------------|
| RQ1 | When do deep probes (MLP / BiLSTM / attention) actually beat a linear probe on entropy signals? | `detector.py`: BiLSTM (~0.78 AUROC) currently *underperforms* LogReg (~0.91). Before investing more in sequence models we want a controlled map of **where** depth pays off. |
| RQ2 | Does multi-pass KL divergence rescue detection of **confident confabulation** (low-entropy hallucinations)? | Root README limitation: *"A model can produce low-entropy hallucinations …"* — single-pass entropy is blind there. The KL signal is the repo's namesake method but was never ablated against this failure mode. |

## How it works — no downloads, no GPUs required

Instead of downloading an LLM, we build a **synthetic pseudo-LM**: a controllable generator of
per-layer next-token logits whose statistics mimic the phenomena the repo measures
(`entropy_baselines.py` features, layer-wise entropy signatures, pass-to-pass instability).
The generative knobs map to real phenomena:

| Knob | Phenomenon |
|------|------------|
| knowledge strength `s` | how grounded the model is in the asked-about entity |
| `gamma` (effect size) | difficulty of the detection problem (how weak the entropy signal is) |
| `confab_frac` | fraction of hallucinations that are *confidently wrong* (low-entropy) |
| per-pass logit noise `sigma(s)` | epistemic instability — larger when knowledge is weaker (MC-dropout analogue) |

All feature extraction reuses the repo's exact conventions (`EPS = 1e-12`, natural-log nats,
the 6D `FEATURE_NAMES` vector from `entropy_baselines.py`, trapezoid AUROC from
`detector.py`) so results transfer conceptually 1:1.

## Logistics

* **Dependencies:** `torch`, `numpy`, `matplotlib` only. Validated with torch 2.11 (CPU), Python 3.14.
* **Runtime:** ~3–5 minutes on CPU.
* **Artifacts:** figures + JSON summaries land in `notebooks/outputs/` when run locally (untracked), or `notebooks/results/` when run via the Kaggle GPU workflow (committed).
* **Seeds:** everything is seeded; reruns are reproducible.

In [ ]:
# Setup — self-contained; deliberately does NOT import repo modules.
import json
import time
from copy import deepcopy
from pathlib import Path

try:
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as exc:
    raise ImportError(
        "PyTorch and NumPy are required for this testbed.\n"
        "Install CPU-only torch with:\n"
        "    pip install torch --index-url https://download.pytorch.org/whl/cpu"
    ) from exc

import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# On a Kaggle kernel, write flat into /kaggle/working/ so `kaggle kernels output`
# downloads a flat file list rather than a nested notebooks/outputs/ subtree.
# Local/Colab runs keep writing to notebooks/outputs/ (untracked scratch dir).
if Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working")
else:
    OUT_DIR = Path("notebooks/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f"PyTorch {torch.__version__} | device: {DEVICE}")
print(f"Artifacts will be written to: {OUT_DIR.resolve()}")

In [ ]:
# ---------------------------------------------------------------------------
# Constants — mirror v2/entropy_baselines.py and v2/detector.py conventions
# ---------------------------------------------------------------------------
EPS = 1e-12                      # same epsilon as entropy_baselines.py
LOGIT_SCALE = 4.0                # kappa: peak logit of an anchored token
NOISE_SCALE = 1.0                # scales the epistemic (per-pass) perturbation

# Simulation geometry
VOCAB = 64                       # pseudo-vocabulary size
SEQ_LEN = 10                     # answer-span length T (token positions)
NUM_LAYERS = 8                   # pseudo-LM depth L
TOP_K = 5                        # matches entropy_baselines.py default top_k
MC_PASSES = 5                    # stochastic forward passes for the KL signal

# Flat 9D feature vector. First 6 names are IDENTICAL to
# v2/entropy_baselines.py::FEATURE_NAMES; the last 3 are divergence extras.
FLAT_FEATURE_NAMES = [
    "entropy_mean",
    "entropy_max",
    "entropy_std",
    "perplexity",
    "topk_entropy_mean",
    "margin_mean",
    "kl_mc_mean",        # NEW: mean pairwise KL across stochastic passes
    "kl_mc_std",         # NEW: std  of pairwise KL across stochastic passes
    "crosslayer_kl_max", # NEW: max over layers of KL(p_l || p_{l+1})
]
ENTROPY_ONLY_SLICE = slice(0, 6)  # view of flat vector WITHOUT KL features

SEQ_FEATURE_NAMES = ["layer_entropy", "layer_margin", "layer_cross_kl"]


def softmax(x: torch.Tensor, axis: int = -1) -> torch.Tensor:
    shifted = x - x.max(dim=axis, keepdim=True).values
    e = torch.exp(shifted)
    return e / (e.sum(dim=axis, keepdim=True) + EPS)


def shannon_entropy(probs: torch.Tensor) -> torch.Tensor:
    # H = -sum p log p over last axis (nats, natural log — repo convention)
    return -(probs * torch.log(probs + EPS)).sum(-1)


def topk_renorm_entropy(probs: torch.Tensor, k: int = TOP_K) -> torch.Tensor:
    topk = probs.topk(k, dim=-1).values
    topk = topk / (topk.sum(-1, keepdim=True) + EPS)
    return -(topk * torch.log(topk + EPS)).sum(-1)


def margin(logits: torch.Tensor) -> torch.Tensor:
    # per-position (top1 - top2) logit gap
    top2 = logits.topk(2, dim=-1).values
    return top2[..., 0] - top2[..., 1]


def kl_div(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    # D_KL(p || q) = sum p (log p - log q), over the last axis
    return (p * (torch.log(p + EPS) - torch.log(q + EPS))).sum(-1)


def compute_auroc(scores, labels) -> float:
    # Port of v2/detector.py::compute_auroc — trapezoid ROC integration.
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    n_pos = float(labels.sum())
    n_neg = float(len(labels) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = np.argsort(-scores, kind="mergesort")
    sl = labels[order]
    tps = np.cumsum(sl) / n_pos
    fps = np.cumsum(1.0 - sl) / n_neg
    tpr = np.concatenate([[0.0], tps])
    fpr = np.concatenate([[0.0], fps])
    trap = getattr(np, "trapezoid", None) or np.trapz
    return float(trap(tpr, fpr))


print(f"vocab={VOCAB} T={SEQ_LEN} layers={NUM_LAYERS} top_k={TOP_K} mc_passes={MC_PASSES}")
print(f"flat features ({len(FLAT_FEATURE_NAMES)}D): {FLAT_FEATURE_NAMES}")
print(f"sequence features per layer: {SEQ_FEATURE_NAMES}")

## 1 · The synthetic pseudo-LM

`generate_dataset` simulates an LLM answering a factual question. For every sample it emits
per-layer next-token logits over the answer span, then extracts features **exactly** as the repo
would from a real HuggingFace model:

* **Flat 9D vector** — the 6D `entropy_baselines.py` vector (`entropy_mean/max/std`,
  `perplexity`, `topk_entropy_mean`, `margin_mean`, computed on the *final layer* over the
  answer span) plus three divergence extras (`kl_mc_mean`, `kl_mc_std`, `crosslayer_kl_max`).
* **Per-layer sequence `(N, L, 3)`** — `[layer_entropy, layer_margin, KL(p_l ‖ p_{l+1})]`,
  the analogue of v2's per-layer attention-family sequences fed to the BiLSTM.

Generative story (all documented simplifications):

1. Each sample draws knowledge strength `s`. Factual samples get `s ~ U[0.75, 1.0]`;
   hallucinated samples get `s = gamma · U[0, 1]`.
2. Logits place mass on a **correct anchor token** with strength `s·profile(l)` and — for
   hallucinations — on a **nearby wrong token**: diffusely for uncertain hallucinations,
   sharply for confident confabulations (controlled by `confab_frac`).
3. Every stochastic pass adds Gaussian logit noise with scale `sigma(s) ∝ (1.15 − s)`:
   weakly-grounded answers are internally unstable even when sharp — the MC-dropout analogue.
   This is the assumption that makes KL divergence informative; it is the sandbox's main
   idealization and is re-examined in Experiment C.

In [ ]:
def generate_dataset(
    n_samples: int,
    gamma: float,
    confab_frac: float,
    seed: int,
    mc_passes: int = MC_PASSES,
) -> dict:
    # --- latents -----------------------------------------------------------
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, 2, (n_samples,), generator=g)
    s = torch.empty(n_samples)
    n_pos = int((y == 1).sum())
    n_neg = n_samples - n_pos
    s[y == 0] = 0.75 + 0.25 * torch.rand(n_neg, generator=g)
    s[y == 1] = gamma * torch.rand(n_pos, generator=g)

    is_hallu = y == 1
    is_confab = is_hallu & (torch.rand(n_samples, generator=g) < confab_frac)
    is_uncertain = is_hallu & ~is_confab
    groups = torch.zeros(n_samples, dtype=torch.long)   # 0 factual / 1 uncertain / 2 confab
    groups[is_uncertain] = 1
    groups[is_confab] = 2

    # --- anchor tokens -------------------------------------------------------
    correct = torch.randint(0, VOCAB, (n_samples, SEQ_LEN), generator=g)
    step = torch.randint(1, 4, (n_samples, SEQ_LEN), generator=g)
    sign = torch.randint(0, 2, (n_samples, SEQ_LEN), generator=g) * 2 - 1
    wrong = (correct + sign * step) % VOCAB             # plausible-but-wrong neighbour token
    bump_c = F.one_hot(correct, VOCAB).float()          # (N, T, V)
    bump_w = F.one_hot(wrong, VOCAB).float()

    # --- per-layer coefficients ---------------------------------------------
    l_hat = torch.linspace(0.0, 1.0, NUM_LAYERS)
    knowledge_profile = 0.35 + 0.65 * l_hat             # grounded sharpening across depth
    syntax_profile = 0.45 + 0.40 * l_hat                # syntactic fluency grows regardless

    coeff_c = s[:, None] * knowledge_profile[None, :]   # (N, L) mass on correct token
    coeff_w = torch.zeros(n_samples, NUM_LAYERS)        # (N, L) mass on wrong token
    coeff_w[is_uncertain] = ((1.0 - s[is_uncertain]) * 0.7)[:, None] * syntax_profile[None, :]
    if int(is_confab.sum()) > 0:
        # confident confabulation: strong WRONG anchor, weak correct anchor
        coeff_w[is_confab] = 1.05 * syntax_profile[None, :].expand(
            int(is_confab.sum()), NUM_LAYERS)
        coeff_c[is_confab] = coeff_c[is_confab] * 0.25

    sigma = NOISE_SCALE * (1.15 - s)                    # epistemic noise scale

    # --- base logits (N, L, T, V) --------------------------------------------
    z_base = LOGIT_SCALE * (
        coeff_c[:, :, None, None] * bump_c[:, None]
        + coeff_w[:, :, None, None] * bump_w[:, None]
    )
    z_base = z_base + 0.15 * torch.randn(z_base.shape, generator=g)

    # --- stochastic forward passes (MC-dropout analogue) ---------------------
    kl_pairs = []
    z0 = p0 = prev_p = None
    for k in range(mc_passes):
        z_k = z_base + sigma[:, None, None, None] * torch.randn(z_base.shape, generator=g)
        p_k = softmax(z_k, axis=-1)
        if k == 0:
            z0, p0 = z_k, p_k                           # the "single pass"
        else:
            kl_pairs.append(kl_div(p_k[:, -1], prev_p[:, -1]).mean(-1))
            kl_pairs.append(kl_div(prev_p[:, -1], p_k[:, -1]).mean(-1))
        prev_p = p_k
    kl_stack = torch.stack(kl_pairs, dim=0)             # (2*(K-1), N)

    # --- flat features from the single pass (final layer, answer span) -------
    p_final = p0[:, -1]                                 # (N, T, V)
    h_tok = shannon_entropy(p_final)                    # (N, T)
    realized = z0[:, -1].argmax(-1)                     # greedy-decoded answer tokens
    nll = -torch.log(
        p_final.gather(-1, realized[..., None]).squeeze(-1) + EPS
    )
    flat = torch.stack([
        h_tok.mean(1),
        h_tok.max(1).values,
        h_tok.std(1, correction=0),
        torch.exp(nll.mean(1)),                         # perplexity of own answer
        topk_renorm_entropy(p_final, TOP_K).mean(1),
        margin(z0[:, -1]).mean(1),
        kl_stack.mean(0),
        kl_stack.std(0, correction=0),
    ], dim=1)

    # --- cross-layer KL on the single pass ------------------------------------
    kl_layers = torch.stack(
        [kl_div(p0[:, l], p0[:, l + 1]).mean(-1) for l in range(NUM_LAYERS - 1)], dim=1,
    )                                                   # (N, L-1)
    kl_layers_full = torch.cat(
        [kl_layers, torch.zeros(n_samples, 1)], dim=1)  # pad to L

    seq = torch.stack([
        shannon_entropy(p0).mean(-1),                   # H_l      (N, L)
        margin(z0).mean(-1),                            # margin_l (N, L)
        kl_layers_full,                                 # KL(l||l+1)
    ], dim=-1)                                          # (N, L, 3)

    flat = torch.cat([flat, kl_layers.max(dim=1).values[:, None]], dim=1)

    return {
        "y": y,
        "flat": flat,              # (N, 9)
        "seq": seq,                # (N, L, 3)
        "groups": groups,          # 0 factual / 1 uncertain-halluc / 2 confab-halluc
        "params": {"n_samples": n_samples, "gamma": gamma,
                   "confab_frac": confab_frac, "seed": seed},
    }


print("generate_dataset defined")

In [ ]:
# Sanity: shapes, finiteness, balance, and that the signals move in the
# direction the information-theoretic story predicts.
set_seed(0)
DS = generate_dataset(n_samples=1200, gamma=0.55, confab_frac=0.35, seed=0)

y, flat, seq = DS["y"], DS["flat"], DS["seq"]
assert flat.shape == (1200, 9) and seq.shape == (1200, NUM_LAYERS, 3), "bad shapes"
assert torch.isfinite(flat).all() and torch.isfinite(seq).all(), "NaN/Inf in features"
balance = y.float().mean().item()
assert 0.4 < balance < 0.6, f"class imbalance: {balance:.2f}"

GROUP_NAMES = ["factual", "uncertain-halluc", "confident-confab"]
print(f"dataset: N=1200 | positive rate {balance:.2f} | groups:",
      {GROUP_NAMES[i]: int((DS['groups'] == i).sum()) for i in range(3)})

rows = []
for gi, name in enumerate(GROUP_NAMES):
    m = DS["groups"] == gi
    rows.append((name,
                 flat[m][:, FLAT_FEATURE_NAMES.index("entropy_mean")].mean().item(),
                 flat[m][:, FLAT_FEATURE_NAMES.index("perplexity")].mean().item(),
                 flat[m][:, FLAT_FEATURE_NAMES.index("kl_mc_mean")].mean().item()))
print()
print(f"{'group':<20} {'entropy_mean':>13} {'perplexity':>11} {'kl_mc_mean':>11}")
for name, e, p, k in rows:
    print(f"{name:<20} {e:>13.3f} {p:>11.3f} {k:>11.3f}")

h_fact = flat[DS['groups'] == 0][:, 0].mean()
h_hallu = flat[y == 1][:, 0].mean()
assert h_hallu > h_fact, "expected higher single-pass entropy for hallucinations"
print("\nsignal direction check passed: hallucinated entropy > factual entropy")

# Figure 1 — per-layer entropy signature per group (the layer-wise divergence story)
fig1, ax = plt.subplots(figsize=(8, 5))
layer_axis = np.arange(1, NUM_LAYERS + 1)
styles = ["tab:blue", "tab:orange", "tab:red"]
for gi, name in enumerate(GROUP_NAMES):
    curve = seq[DS["groups"] == gi][..., SEQ_FEATURE_NAMES.index("layer_entropy")].mean(0)
    ax.plot(layer_axis, curve.numpy(), "-o", color=styles[gi], label=name)
ax.set_xlabel("pseudo-LM layer")
ax.set_ylabel("mean token entropy (nats)")
ax.set_title("Per-layer entropy signatures (gamma=0.55, confab_frac=0.35)")
ax.legend()
ax.grid(alpha=0.3)
fig1.tight_layout()
fig1.savefig(OUT_DIR / "fig1_layer_entropy_signatures.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved -> {OUT_DIR / 'fig1_layer_entropy_signatures.png'}")

## 2 · Probe architectures (pure `torch.nn`)

Four probes, all consuming standardized features and emitting raw logits:

| Probe | Input | Notes |
|-------|-------|-------|
| `LogRegTorch` | flat `(N, D)` | linear control — the v2 champion baseline |
| `MLPProbe` | flat `(N, D)` | nonlinear feature interactions (v2's secondary model) |
| `BiLSTMProbe` | sequence `(N, L, 3)` | mirrors `v2/detector.py` BiLSTMDetector recipe |
| `AttnProbe` | sequence `(N, L, 3)` | **candidate for v3** — self-attention over layers + mean-pool head |

In [ ]:
class LogRegTorch(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.lin = nn.Linear(in_dim, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)


class MLPProbe(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 32, p_drop: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class BiLSTMProbe(nn.Module):
    def __init__(self, in_dim: int = 3, hidden: int = 16,
                 num_layers: int = 1, p_drop: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            in_dim, hidden, num_layers=num_layers, batch_first=True,
            bidirectional=True,
            dropout=p_drop if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(2 * hidden, 1))

    def forward(self, x):                       # x: (B, L, F)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class AttnProbe(nn.Module):
    def __init__(self, in_dim: int = 3, d_model: int = 24, heads: int = 2,
                 p_drop: float = 0.2, max_len: int = 64):
        super().__init__()
        self.inp = nn.Linear(in_dim, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, std=0.02)
        self.attn = nn.MultiheadAttention(
            d_model, heads, dropout=p_drop, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(d_model, 1))

    def forward(self, x):                       # x: (B, L, F)
        _, L, _ = x.shape
        h = self.inp(x) + self.pos[:, :L]
        a, _ = self.attn(h, h, h, need_weights=False)
        h = self.norm(h + a)
        return self.head(h.mean(dim=1)).squeeze(-1)


ARCH_FACTORIES = {
    "logreg": LogRegTorch,
    "mlp": MLPProbe,
    "bilstm": lambda in_dim: BiLSTMProbe(in_dim=in_dim),
    "attn": lambda in_dim: AttnProbe(in_dim=in_dim),
}
SEQ_ARCHS = {"bilstm", "attn"}
print("architectures:", list(ARCH_FACTORIES))

In [ ]:
def standardize(train_x: torch.Tensor, *others: torch.Tensor):
    # z-score using TRAIN statistics only (mirrors v2 detector _mean/_std pattern)
    flat_train = train_x.reshape(-1, train_x.shape[-1])
    mean = flat_train.mean(dim=0)
    std = flat_train.std(dim=0, correction=0).clamp(min=1e-8)
    def apply(x):
        return ((x - mean) / std).float()
    return [apply(x) for x in (train_x, *others)]


def stratified_split(y: torch.Tensor, seed: int, val_frac: float = 0.2,
                     test_frac: float = 0.34):
    rng = np.random.default_rng(seed)
    y_np = y.numpy()
    parts = []
    for cls in (0, 1):
        idx = np.where(y_np == cls)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_frac))
        n_val = int(round(len(idx) * val_frac))
        parts.append((idx[n_test + n_val:], idx[n_test:n_test + n_val], idx[:n_test]))
    tr = np.concatenate([p[0] for p in parts])
    va = np.concatenate([p[1] for p in parts])
    te = np.concatenate([p[2] for p in parts])
    for arr in (tr, va, te):
        np.random.default_rng(seed).shuffle(arr)
    return tr, va, te


def eval_auroc(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():
        logits = model(X.to(DEVICE))
        scores = torch.sigmoid(logits).cpu().numpy()
    return compute_auroc(scores, y.numpy())


def train_probe(build_model, Xtr, ytr, Xva, yva, *, max_epochs: int = 80,
                batch_size: int = 64, lr: float = 1e-3, weight_decay: float = 1e-4,
                patience: int = 12, seed: int = 0):
    # AdamW + BCEWithLogitsLoss + grad clip 1.0 + early stop on val AUROC
    set_seed(seed)
    in_dim = Xtr.shape[-1]
    model = build_model(in_dim).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    lossf = nn.BCEWithLogitsLoss()

    Xtr_d, ytr_d = Xtr.to(DEVICE), ytr.float().to(DEVICE)
    Xva_d = Xva.to(DEVICE)
    gen = torch.Generator().manual_seed(seed + 1)

    best_auc, best_state, best_ep = -1.0, None, -1
    history = []
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xtr_d), generator=gen)
        for i in range(0, len(perm), batch_size):
            b = perm[i:i + batch_size]
            opt.zero_grad()
            loss = lossf(model(Xtr_d[b]), ytr_d[b])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        auc = eval_auroc(model, Xva_d, yva)
        history.append(auc)
        if auc > best_auc:
            best_auc, best_state, best_ep = auc, deepcopy(model.state_dict()), epoch
        elif epoch - best_ep >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    return model, {"best_val_auroc": best_auc, "best_epoch": best_ep,
                   "epochs_run": len(history)}


def run_single(ds: dict, arch: str, seed: int, feature_set: str = "full",
               **train_kwargs):
    # feature_set: "full" (9D) | "entropy_only" (6D) | "seq" (N, L, 3)
    tr, va, te = stratified_split(ds["y"], seed)
    if arch in SEQ_ARCHS or feature_set == "seq":
        X = ds["seq"]
    else:
        sl = ENTROPY_ONLY_SLICE if feature_set == "entropy_only" else slice(None)
        X = ds["flat"][:, sl]
    y = ds["y"]

    Xtr, Xva, Xte = standardize(X[tr], X[va], X[te])
    model, info = train_probe(
        ARCH_FACTORIES[arch], Xtr, y[tr], Xva, y[va], seed=seed, **train_kwargs)
    test_auc = eval_auroc(model, Xte, y[te])
    return {"test_auroc": test_auc, **info}


# smoke test: one tiny run of each architecture end-to-end
for arch in ARCH_FACTORIES:
    r = run_single(DS, arch, seed=0, max_epochs=5)
    print(f"smoke {arch:>7}: val {r['best_val_auroc']:.3f} | "
          f"test {r['test_auroc']:.3f} | stopped @ epoch {r['best_epoch']}")
print("harness OK")

## 3 · Experiment A — architecture bake-off (RQ1)

Moderate difficulty (`gamma = 0.55`, 35% confident confabulation). Each architecture is trained
with 3 seeds on the same splits; we report test AUROC mean ± std.

**What would change our minds:** if `bilstm`/`attn` beat `logreg` here but lose in the
Experiment B sweep at realistic effect sizes, sequence models are only worth it for hard regimes.

In [ ]:
set_seed(42)
ARCHS = ["logreg", "mlp", "bilstm", "attn"]
SEEDS_A = [0, 1, 2]

exp_a = {}
t0 = time.time()
print(f"{'arch':>7} | {'AUROC (mean ± std)':>20} | per-seed")
print("-" * 60)
for arch in ARCHS:
    aucs = []
    for sd in SEEDS_A:
        r = run_single(DS, arch, seed=sd)
        aucs.append(r["test_auroc"])
    exp_a[arch] = {"mean": float(np.mean(aucs)),
                   "std": float(np.std(aucs)),
                   "per_seed": [float(a) for a in aucs]}
    print(f"{arch:>7} | {np.mean(aucs):>12.3f} ± {np.std(aucs):.3f} | "
          f"{['%.3f' % a for a in aucs]}")
elapsed_a = time.time() - t0
print(f"\n[{elapsed_a:.1f}s]")

with open(OUT_DIR / "expA_architecture.json", "w") as f:
    json.dump({"results": exp_a, "dataset_params": DS["params"]}, f, indent=2)

fig2, ax = plt.subplots(figsize=(7, 4.5))
means = [exp_a[a]["mean"] for a in ARCHS]
stds = [exp_a[a]["std"] for a in ARCHS]
bars = ax.bar(ARCHS, means, yerr=stds, capsize=5,
              color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
ax.axhline(0.5, ls="--", c="gray", lw=1, label="chance")
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("test AUROC")
ax.set_title("Experiment A — architecture comparison (gamma=0.55)")
ax.legend()
fig2.tight_layout()
fig2.savefig(OUT_DIR / "fig2_experiment_a.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved -> {OUT_DIR / 'fig2_experiment_a.png'} and expA_architecture.json")

## 4 · Experiment B — where does depth pay off? (RQ1)

We sweep the knowledge gap `gamma` from *easy* (hallucinations are wildly uncertain) to *hard*
(hallucinations nearly as confident as facts). For each difficulty we regenerate a fresh dataset
and train every architecture with 2 seeds.

**Reading the plot:** if sequence models only win in the low-`gamma` tail, v3 should keep a linear
probe as the default and deploy deep probes only when calibration data suggests weak entropy signal.

In [ ]:
set_seed(43)
GAMMAS = [0.15, 0.30, 0.45, 0.60, 0.80, 1.00]
SEEDS_B = [0, 1]

exp_b = {arch: np.zeros((len(GAMMAS), len(SEEDS_B))) for arch in ARCHS}
t0 = time.time()
for gi, gam in enumerate(GAMMAS):
    ds_g = generate_dataset(n_samples=1200, gamma=gam,
                            confab_frac=DS["params"]["confab_frac"], seed=100 + gi)
    for ai, arch in enumerate(ARCHS):
        for si, sd in enumerate(SEEDS_B):
            r = run_single(ds_g, arch, seed=sd)
            exp_b[arch][gi, si] = r["test_auroc"]
    print(f"gamma={gam:.2f} done  [{time.time() - t0:.1f}s]")

exp_b_json = {
    arch: {"gammas": GAMMAS,
           "mean": exp_b[arch].mean(axis=1).tolist(),
           "all_runs": exp_b[arch].tolist()}
    for arch in ARCHS
}
with open(OUT_DIR / "expB_gamma_sweep.json", "w") as f:
    json.dump(exp_b_json, f, indent=2)

fig3, ax = plt.subplots(figsize=(8, 5))
colors = dict(zip(ARCHS, ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]))
for arch in ARCHS:
    m = exp_b[arch].mean(axis=1)
    lo = exp_b[arch].min(axis=1)
    hi = exp_b[arch].max(axis=1)
    ax.plot(GAMMAS, m, "-o", color=colors[arch], label=arch)
    ax.fill_between(GAMMAS, lo, hi, color=colors[arch], alpha=0.15)
ax.axhline(0.5, ls="--", c="gray", lw=1)
ax.set_xlabel("knowledge gap gamma  (low = harder detection)")
ax.set_ylabel("test AUROC")
ax.set_title("Experiment B — detection difficulty sweep")
ax.legend()
ax.grid(alpha=0.3)
fig3.tight_layout()
fig3.savefig(OUT_DIR / "fig3_gamma_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'gamma':>6} | " + " | ".join(f"{a:>7}" for a in ARCHS))
print("-" * 45)
for gi, gam in enumerate(GAMMAS):
    print(f"{gam:>6.2f} | " + " | ".join(
        f"{exp_b[a][gi].mean():>7.3f}" for a in ARCHS))
print(f"\nsaved -> fig3_gamma_sweep.png, expB_gamma_sweep.json")

## 5 · Experiment C — confident confabulation stress test (RQ2)

The repo's documented failure mode: hallucinations that are **sharp** (low entropy) but wrong.
We regenerate datasets with `confab_frac` = 0% (honest uncertainty), 50%, 100% (all confident
confabulations) and compare detectors:

| Detector | Input | Question |
|----------|-------|----------|
| LogReg · 6D | single-pass entropy only | the established baseline |
| LogReg · 9D | + KL divergence features | **does MC-pass KL rescue detection?** |
| MLP · 9D | + KL divergence features | nonlinear interactions |
| BiLSTM / Attn · seq | per-layer sequences | do layer dynamics help? |

If AUROC for the 6D detector collapses as `confab_frac → 1` while the 9D detectors degrade
gracefully, the multi-pass divergence signal is earning its compute cost.

In [ ]:
set_seed(44)
REGIMES = [("honest_uncertainty", 0.0), ("mixed", 0.5), ("full_confabulation", 1.0)]
DETECTORS_C = [
    ("logreg", "entropy_only", "LogReg · 6D entropy"),
    ("logreg", "full",         "LogReg · 9D +KL"),
    ("mlp",    "full",         "MLP · 9D +KL"),
    ("bilstm", "seq",          "BiLSTM · layer seq"),
    ("attn",   "seq",          "Attn · layer seq"),
]
SEEDS_C = [0, 1]

exp_c = {name: {} for _, _, name in DETECTORS_C}
t0 = time.time()
for regime_name, cfrac in REGIMES:
    ds_c = generate_dataset(n_samples=1200, gamma=DS["params"]["gamma"],
                            confab_frac=cfrac, seed=200 + int(cfrac * 10))
    print(f"regime {regime_name:<20} "
          f"(mean entropy: halluc={ds_c['flat'][ds_c['y'] == 1][:, 0].mean():.2f} nats)")
    for arch, feat_set, label in DETECTORS_C:
        aucs = [run_single(ds_c, arch, seed=sd,
                           feature_set=feat_set)["test_auroc"] for sd in SEEDS_C]
        exp_c[label][regime_name] = float(np.mean(aucs))
        print(f"   {label:<22} AUROC {np.mean(aucs):.3f}")
print(f"[{time.time() - t0:.1f}s]")

with open(OUT_DIR / "expC_confab_stress.json", "w") as f:
    json.dump(exp_c, f, indent=2)

labels = [lbl for _, _, lbl in DETECTORS_C]
x = np.arange(len(REGIMES))
width = 0.15
fig4, ax = plt.subplots(figsize=(9.5, 5))
for di, lbl in enumerate(labels):
    vals = [exp_c[lbl][rname] for rname, _ in REGIMES]
    ax.bar(x + (di - 2) * width, vals, width, label=lbl)
ax.set_xticks(x)
ax.set_xticklabels([f"{rn}\n(confab_frac={cf:.0%})" for rn, cf in REGIMES])
ax.axhline(0.5, ls="--", c="gray", lw=1)
ax.set_ylim(0.3, 1.0)
ax.set_ylabel("test AUROC")
ax.set_title("Experiment C — robustness to confident confabulation")
ax.legend(fontsize=8, loc="lower left")
fig4.tight_layout()
fig4.savefig(OUT_DIR / "fig4_confab_stress.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved -> fig4_confab_stress.png, expC_confab_stress.json")

## 6 · Findings & integration checklist

**Fill in after running** (numbers land in `notebooks/outputs/*.json`):

* **RQ1 — architecture:** `logreg` vs `mlp` vs `bilstm` vs `attn` in Exp A; the gamma sweep
  (Exp B) shows *where* depth pays off. If deep probes only win at low `gamma`, keep LogReg as
  the production default and treat sequence probes as a hard-regime tool.
* **RQ2 — divergence signal:** compare "LogReg · 6D" against "LogReg · 9D +KL" across Exp C
  regimes. A shrinking gap (or reversal) as `confab_frac` grows supports adding multi-pass KL
  features to the real feature extractor.

### Porting checklist (only if experiments justify it)

| Finding | Integration target |
|---------|--------------------|
| Sequence probes beat flat probes on hard regimes | add `classifier_type="attn"` to `v2/detector.py::HallucinationDetector` |
| KL features rescue confabulation detection | extend `v2/entropy_baselines.py` with a multi-pass variant (needs N stochastic forward passes → cost note in README) |
| Per-layer entropy signature is discriminative | revisit root-README's aspirational `LayerEntropyAnalyzer` with a real design |
| Effect-size crossover mapped | document in `paper/` as guidance for labeling-budget allocation |

### Caveats — read before citing any number here

1. The pseudo-LM is an **idealization**: real models couple sharpness and instability differently,
   and RLHF compresses entropy distributions (open question #1 in the root README).
2. The KL signal's informativeness here rests on the assumed `sigma(s) ∝ (1 − s)` noise scaling.
   Experiment C measures robustness *given* that assumption; validating it empirically requires
   re-running on real model logprobs (e.g., export features from the v2 pipeline on HaluEval).
3. Sample sizes are small (N=1200, 2–3 seeds) — this is directional evidence for triage, not
   paper-grade statistics.

### Housekeeping

* This notebook and `notebooks/outputs/` are **not committed**; nothing was staged or pushed.
* Before committing anything derived from this sandbox: clear outputs (`Cell → All Output → Clear`),
  decide which findings graduate into `v2/`, and update the status checklist in the root README.